# Fish Speech

### For Windows User / win用户

In [ ]:
!chcp 65001

### For Linux User / Linux 用户

In [ ]:
import locale
locale.setlocale(locale.LC_ALL, 'en_US.UTF-8')

### Prepare Model

In [ ]:
# For Chinese users, you probably want to use mirror to accelerate downloading
# !set HF_ENDPOINT=https://hf-mirror.com
# !export HF_ENDPOINT=https://hf-mirror.com 

!hf download fishaudio/openaudio-s1-mini --local-dir checkpoints/openaudio-s1-mini/

## WebUI Inference

> You can use --compile to fuse CUDA kernels for faster inference (10x).

In [2]:
!python tools/run_webui.py \
    --llama-checkpoint-path checkpoints/openaudio-s1-mini \
    --decoder-checkpoint-path checkpoints/openaudio-s1-mini/codec.pth \
    --compile

Traceback (most recent call last):
  File "/home/fish-speech-s1/tools/run_webui.py", line 6, in <module>
    import torch
  File "/root/miniconda3/envs/fish-s1/lib/python3.12/site-packages/torch/__init__.py", line 2229, in <module>
    from torch import _VF as _VF, functional as functional  # usort: skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/miniconda3/envs/fish-s1/lib/python3.12/site-packages/torch/functional.py", line 8, in <module>
    import torch.nn.functional as F
  File "/root/miniconda3/envs/fish-s1/lib/python3.12/site-packages/torch/nn/__init__.py", line 8, in <module>
    from torch.nn.modules import *  # usort: skip # noqa: F403
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/miniconda3/envs/fish-s1/lib/python3.12/site-packages/torch/nn/modules/__init__.py", line 2, in <module>
    from .linear import Bilinear, Identity, LazyLinear, Linear  # usort: skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/miniconda

## Break-down CLI Inference

### 1. Encode reference audio: / 从语音生成 prompt: 

You should get a `fake.npy` file.

你应该能得到一个 `fake.npy` 文件.

In [1]:
## Enter the path to the audio file here
src_audio = r"/home/fish-speech-s1/assert/andelie.wav"

!python fish_speech/models/dac/inference.py \
    -i {src_audio} \
    --checkpoint-path "checkpoints/openaudio-s1-mini/codec.pth"

from IPython.display import Audio, display
audio = Audio(filename="fake.wav")
display(audio)

/root/miniconda3/envs/fish-s1/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
2026-03-24 11:11:31.249 | INFO     | __main__:load_model:46 - Loaded model: <All keys matched successfully>
2026-03-24 11:11:31.250 | INFO     | __main__:main:75 - Processing in-place reconstruction of /home/fish-speech-s1/assert/andelie.wav
Traceback (most recent call last):
  File "/home/fish-speech-s1/fish_speech/models/dac/inference.py", line 123, in <module>
    main()
  File "/root/miniconda3/envs/fish-s1/lib/python3.12/site-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/root/miniconda3/envs/fish-s1/lib/python3.12/site-packages/click/core.py", line 1485, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^


### 2. Generate semantic tokens from text: / 从文本生成语义 token:

> This command will create a codes_N file in the working directory, where N is an integer starting from 0.

> You may want to use `--compile` to fuse CUDA kernels for faster inference (~30 tokens/second -> ~300 tokens/second).

> 该命令会在工作目录下创建 codes_N 文件, 其中 N 是从 0 开始的整数.

> 您可以使用 `--compile` 来融合 cuda 内核以实现更快的推理 (~30 tokens/秒 -> ~300 tokens/秒)

In [6]:
!python fish_speech/models/text2semantic/inference.py \
    --text "hello world" \
    --prompt-text "Здравствуйте, я Ханна, ваш автомобильный АИ помощник.Я помогу узнать погоду, построить маршрут, включить музыку и сделать поездку удобнее и безопаснее.Всегда рядом, чтобы помочь вам в дороге." \
    --prompt-tokens "fake.npy" \
    --checkpoint-path "checkpoints/openaudio-s1-mini" \
    --num-samples 2 \
    --compile

2026-03-24 10:34:28.482 | INFO     | __main__:main:644 - Loading model ...
You are using a model of type `dual_ar` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
2026-03-24 10:34:28.484 | WARNING  | fish_speech.models.text2semantic.llama:from_pretrained:508 - Failed to load tokenizer for config injection: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.. Semantic IDs might be 0.
2026-03-24 10:34:28.484 | INFO     | fish_speech.models.text2semantic.ll

### 3. Generate speech from semantic tokens: / 从语义 token 生成人声:

In [ ]:
!python fish_speech/models/dac/inference.py \
    -i "codes_0.npy" \
    --checkpoint-path "checkpoints/openaudio-s1-mini/codec.pth"

from IPython.display import Audio, display
audio = Audio(filename="fake.wav")
display(audio)